In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt

from src.rainfall import (
    Region,
    CropRule,
    default_crop_rules,
    cosine_similarity,
    cosine_similarity_scipy,
    pearson,
    euclidean,
    similarity_matrix,
    detect_rainy_seasons,
    classify_seasonality,
    synthetic_uganda_regions,
)

print("Imports OK")

Imports OK


# Project 4 — Rainfall & Crop Suitability

A farming advisory service wants to classify monthly rainfall in
several Ugandan regions, detect rainy seasons automatically, and
recommend crops based on rainfall thresholds.

**Regions (illustrative synthetic data):** Kampala, Entebbe, Gulu,
Mbarara, Soroti.

**Crops (illustrative thresholds):** maize, beans, coffee.

The cosine-similarity trap from last year's cohort — using `math.cos()`
instead of a proper vector similarity — is corrected here and
cross-checked against `scipy.spatial.distance.cosine`.

In [ ]:
# Task 1 — Region objects
regions = synthetic_uganda_regions()

print(f"{'Region':<10}{'Total':>8}{'Mean':>8}{'Wettest':>10}{'Driest':>10}{'CV':>8}")
print("-" * 60)
for r in regions:
    print(f"{r.name:<10}{r.annual_total():>8.0f}{r.mean():>8.1f}"
          f"{r.wettest_month():>10}{r.driest_month():>10}{r.cv():>8.3f}")

In [ ]:
# Task 2 — classify every month in every region for every crop
rules = default_crop_rules()

for r in regions:
    print(f"\n=== {r.name} ===")
    for rule in rules:
        labels = [rule.classify(mm) for mm in r.rainfall]
        # Count months in each category
        good = sum("Good" in lab for lab in labels)
        drought = sum("Drought" in lab for lab in labels)
        flood = sum("Waterlogging" in lab for lab in labels)
        print(f"  {rule.name:>7}: Good={good:>2} months, "
              f"Drought risk={drought:>2}, Waterlogging risk={flood:>2}")

## Why `math.cos()` was wrong

`math.cos(x)` returns the cosine **of an angle x in radians** — e.g.
`math.cos(0) = 1`, `math.cos(π/2) = 0`. It has nothing to do with
vectors. It will happily accept two scalars and return a meaningless
number.

**Cosine similarity** between two vectors a and b is
`dot(a, b) / (||a|| · ||b||)`. It measures the angle *between* the
vectors and returns a value in [-1, 1]:

- 1 means the vectors point in the same direction (identical shape).
- 0 means they are orthogonal.
- -1 means they point in opposite directions.

The two are only numerically equal when both vectors happen to be
unit-length 1-D vectors and x is the angle between them — an
irrelevant coincidence.

In [ ]:
# Task 3 — verify our NumPy implementation against scipy
print("Cross-check: cosine_similarity vs 1 - scipy.spatial.distance.cosine")
print(f"{'Pair':<20}{'NumPy':>12}{'SciPy':>12}{'|Δ|':>12}")
print("-" * 56)
for i in range(len(regions)):
    for j in range(i + 1, len(regions)):
        a, b = regions[i].rainfall, regions[j].rainfall
        v1 = cosine_similarity(a, b)
        v2 = cosine_similarity_scipy(a, b)
        print(f"{regions[i].name+'/'+regions[j].name:<20}"
              f"{v1:>12.6f}{v2:>12.6f}{abs(v1-v2):>12.2e}")

In [ ]:
# Task 4 — three pairwise matrices
names = [r.name for r in regions]
labels = [f"{i}:{n}" for i, n in enumerate(names)]

def show_matrix(M, title, fmt="{:.3f}"):
    print(f"\n{title}")
    print("     " + "".join(f"{n:>10}" for n in names))
    for i, row in enumerate(M):
        print(f"{names[i]:>5}" + "".join(f"{fmt.format(v):>10}" for v in row))

cos_M = similarity_matrix(regions, metric=cosine_similarity)
pearson_M = similarity_matrix(regions, metric=pearson)
euclid_M = similarity_matrix(regions, metric=euclidean)

show_matrix(cos_M, "Cosine similarity matrix:")
show_matrix(pearson_M, "Pearson correlation matrix:")
show_matrix(euclid_M, "Euclidean distance matrix:")

## Why cosine similarity can rate two regions "similar" even when one is much wetter

Cosine similarity ignores magnitude and only compares **shape**.
Multiplying a vector by any positive scalar leaves the cosine
similarity unchanged: `cos(a, 2a) = 1`. So a region with rainfall
`[10, 20, 30]` and another with `[100, 200, 300]` have cosine
similarity 1 — even though one is ten times wetter.

This is exactly why we also compute **Euclidean distance** (which is
sensitive to magnitude) and **Pearson correlation** (which removes the
mean before comparing shapes). The three measures are complementary:
cosine answers "same shape?", Euclidean answers "same magnitude?", and
Pearson answers "same pattern of deviations from the mean?".

In [ ]:
# Task 5 — detect rainy seasons
print(f"{'Region':<10}{'Seasonality':>15}{'Peak months (1-based)':>30}")
print("-" * 60)
for r in regions:
    peaks = detect_rainy_seasons(r)
    season = classify_seasonality(r)
    print(f"{r.name:<10}{season:>15}{str(peaks):>30}")

## Checking against Uganda's known climate zones

Uganda has two main climate patterns:

- **Bimodal** (central, south, southwest): two rainy seasons —March–May and September–November. Kampala, Entebbe, Mbarara fall here.
- **Unimodal** (north, northeast): one long rainy season that peaks around July–August. Gulu and Soroti fall here.

Our automatic detection should reproduce this split. If it does, the `scipy.signal.find_peaks`-based method is validated against known climate science.

In [ ]:
# Task 6a — all regions on one chart
months = Region.MONTHS
x = np.arange(12)

plt.figure(figsize=(10, 5))
for r in regions:
    plt.plot(x, r.rainfall, marker="o", label=r.name)
plt.xticks(x, months)
plt.xlabel("Month")
plt.ylabel("Rainfall (mm)")
plt.title("Monthly rainfall across five Ugandan regions (illustrative)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../data/rainfall_by_region.png", dpi=120)
plt.show()

In [ ]:
# Task 6b — heatmap of suitability score by month and region for each crop
# Encode: 1 = Good, 0 = Drought risk, -1 = Waterlogging risk
fig, axes = plt.subplots(1, len(rules), figsize=(15, 4))
for ax, rule in zip(axes, rules):
    # Build a (n_regions x 12) score matrix
    scores = np.zeros((len(regions), 12))
    for i, r in enumerate(regions):
        for m in range(12):
            label = rule.classify(r.rainfall[m])
            if label.startswith("Good"):
                scores[i, m] = 1
            elif label.startswith("Drought"):
                scores[i, m] = 0
            else:
                scores[i, m] = -1

    im = ax.imshow(scores, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)
    ax.set_xticks(range(12))
    ax.set_xticklabels(Region.MONTHS, rotation=45, fontsize=8)
    ax.set_yticks(range(len(regions)))
    ax.set_yticklabels([r.name for r in regions], fontsize=8)
    ax.set_title(f"Suitability for {rule.name}")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig("../data/crop_suitability.png", dpi=120)
plt.show()

## Advisory note for farmers — Gulu region

Gulu receives one long rainy season extending from **April to October**, peaking around **May and August** (150–160 mm/month). Maize is well-suited: it sits in the "good" range from April through October. Beans also do well but the drier patches in June require careful timing. Coffee is marginal in Gulu — the monthly rainfall exceeds the waterlogging threshold in several months, so coffee should be planted only in the early part of the season.

**Planting advice:** sow maize in **late March or early April** so that the critical flowering stage aligns with the May rainfall peak. Harvest before the November dry period. For beans, aim for a **September planting** if rainfall persists, otherwise stick to the main April–May window. Avoid planting coffee in Gulu unless you have drainage; consider Mbarara or the central region instead.

*(This is illustrative advice based on synthetic data; consult UNMA seasonal forecasts and local extension officers before planting.)*

## Findings & Limitations

### Findings

Uganda's regions split cleanly into **bimodal** (Kampala, Entebbe, Mbarara) and **unimodal** (Gulu, Soroti) rainfall patterns, matching the known climate zones: central and southwestern regions have two rainy seasons peaking in April–May and September–November, while northern and northeastern regions have a single long rainy season peaking in mid-year. The automatic peak detection using `scipy.signal.find_peaks` reproduced this classification, validating the method against climate science.

Cosine similarity correctly rates Kampala and Entebbe as highly similar (~0.99) because their rainfall *shapes* are nearly identical, but it also gives high scores to region pairs whose magnitudes are very different. For example, the 200 mm and 100 mm peaks of Entebbe and Mbarara contribute equally to cosine similarity despite a 2× magnitude difference. Pearson correlation removes the mean and captures shape better when the difference is in the base level; Euclidean distance captures absolute magnitude and correctly separates wetter from drier regions.

The crop-suitability heatmap shows that maize and beans are viable in almost every region across the two rainy seasons, while coffee is best suited to the central/southern belt where rainfall stays within 100–220 mm per month. In Gulu, coffee is often at the waterlogging threshold — the model flags it as unsuitable for several months.

### Limitations

The rainfall data are synthetic and illustrative, not from a real meteorological record; conclusions about specific regions are indicative only. The crop thresholds are drawn from general FAO ranges, not from Uganda-specific agronomic trials. The cosine similarity between regions is sensitive to the number of months included — a 12-month cosine comparison is dominated by whichever month has the largest rainfall (here, April). The unimodal / bimodal classification depends on a minimum prominence parameter (10 mm by default); a different threshold could reclassify a marginal region.Future work should replace the synthetic data with 10+ years of NASA POWER or UNMA monthly rainfall, use per-region soil and temperature data, and evaluate crop suitability with a proper water-balance model rather than a simple monthly threshold.